# Collect data for PILOT

In [23]:
from pathlib import Path
import pandas as pd

# Поддерживаем запуск из data/pilot или корня проекта.
candidates = [Path.cwd(), Path.cwd() / "data" / "pilot"]
pilot_dir = next((p for p in candidates if (p.parent / "COAD_Mutations.csv").exists()), None)
if pilot_dir is None:
    raise FileNotFoundError("Запустите ноутбук из data/pilot или корня проекта")
data_dir = pilot_dir.parent

COAD_Mutations = pd.read_csv(data_dir / "COAD_Mutations.csv", low_memory=False)
COAD_Expression = pd.read_csv(data_dir / "COAD_Expression.csv")
COAD_CRISPRGeneEffect = pd.read_csv(data_dir / "COAD_CRISPRGeneEffect.csv")
COAD_Model = pd.read_csv(data_dir / "COAD_Model.csv")

display(COAD_Mutations.head(2))
display(COAD_Expression.head(2))
display(COAD_CRISPRGeneEffect.head(2))
display(COAD_Model.head(2))


,Unnamed: 0,SequencingID,ModelID,ModelConditionID,IsDefaultEntryForModel,IsDefaultEntryForMC,Chrom,Pos,Ref,Alt,...,GwasDisease,GwasPmID,GtexGene,ProveanPrediction,AMClass,AMPathogenicity,Rescue,RescueReason,Hotspot,EntrezGeneID
0,20,CDS-vHArJs,ACH-000969,MC-000969-drm8,Yes,Yes,chr1,942225,TG,GC,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,148398.0
1,68,CDS-kmfiCf,ACH-000957,MC-000957-Yckn,Yes,Yes,chr1,961336,GC,G,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,339451.0


,Unnamed: 0,SequencingID,ModelConditionID,ModelID,IsDefaultEntryForMC,IsDefaultEntryForModel,TSPAN6 (7105),TNMD (64102),DPM1 (8813),SCYL3 (57147),...,ATXN8 (724066),SMIM42 (117981789),NPBWR1 (2831),ACTL10 (170487),RNF228 (122319436),PANO1 (101927423),HRURF (120766137),PRRC2B (84726),F8A2 (474383),F8A1 (8263)
0,68,CDS-2HrGbX,MC-000999-JOmJ,ACH-000999,Yes,Yes,4.762301,0.486458,6.793519,2.585237,...,0.0,0.0,0.010904,2.002879,0.000000,0.821495,0.0,6.354614,1.04431,4.363143
1,115,CDS-46Q1Af,MC-000552-bdsQ,ACH-000552,Yes,Yes,4.092850,0.000000,7.012323,3.158833,...,0.0,0.0,0.000000,1.766875,0.015205,0.514804,0.0,6.048471,0.00000,5.173108


,ModelID,A1BG (1),A1CF (29974),A2M (2),A2ML1 (144568),A3GALT2 (127550),A4GALT (53947),A4GNT (51146),AAAS (8086),AACS (65985),...,OR3A2 (4995),OR3A3 (8392),POLR2J (5439),PRAMEF10 (343071),PRR33 (102724536),RGPD2 (729857),SMIM10L3 (122526779),TPRX2 (503627),TTLL13 (440307),VCX2 (51480)
0,ACH-000552,0.100104,0.091508,-0.084647,0.127559,0.104604,0.037875,0.283021,-0.240823,-0.040064,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ACH-000958,-0.165650,0.029107,-0.058589,0.121673,0.025537,-0.276521,0.020071,-0.093698,-0.191400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,ModelID,PatientID,CellLineName,StrippedCellLineName,DepmapModelType,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,OncotreeCode,PatientSubtypeFeatures,...,PublicComments,CCLEName,HCMIID,PediatricModelType,ModelAvailableInDbgap,ModelSubtypeFeatures,WTSIMasterCellID,SangerModelID,COSMICID,ModelIDAlias
0,ACH-000003,PT-puKIyc,CACO2,CACO2,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,CACO2_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM00891,NaN,NaN
1,ACH-000007,PT-NOXwpH,LS513,LS513,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,LS513_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,569.0,SIDM00677,907795.0,ACH-001078


In [24]:
cond = (
    COAD_Mutations["IsDefaultEntryForModel"].eq("Yes")
    & COAD_Mutations["HugoSymbol"].eq("KRAS")
)

kras_variants = COAD_Mutations.loc[
    cond, ["ModelID", "ProteinChange", "Hotspot"]
].copy()

# Явное преобразование: строка "False" не должна стать True.
kras_variants["is_hotspot"] = (
    kras_variants["Hotspot"]
    .astype("string").str.strip().str.lower()
    .map({"true": True, "false": False})
    .astype("boolean")
)

KRAS_changes = (
    COAD_Mutations.loc[cond, ["ModelID", "ProteinChange"]]
    .groupby("ModelID")["ProteinChange"]
    .agg(
        lambda values: "; ".join(
            sorted(values.dropna().unique())
        ) or pd.NA
    )
    .rename("KRAS_protein_changes")
    .reset_index()
)

KRAS_hotspot = (
    kras_variants.loc[kras_variants["is_hotspot"].fillna(False)]
    .groupby("ModelID").size().gt(0).astype("Int64")
    .rename("KRAS_hotspot").reset_index()
)

In [25]:
# Основной профиль каждой модели, только ID и экспрессия KRAS
cond_expression = COAD_Expression["IsDefaultEntryForModel"].eq("Yes")

KRAS_expression = (
    COAD_Expression.loc[
        cond_expression,
        ["ModelID", "KRAS (3845)"]
    ]
    .rename(columns={"KRAS (3845)": "KRAS_expression"})
    .copy()
)

# Для объединения нужна одна строка на модель
assert KRAS_expression["ModelID"].is_unique, \
    "Обнаружено несколько основных профилей для одной модели"

print("Количество линий:", len(KRAS_expression))
display(KRAS_expression.head())

Количество линий: 66


,ModelID,KRAS_expression
0,ACH-000999,4.626281
1,ACH-000552,4.958708
2,ACH-000501,4.141580
3,ACH-000798,4.514318
4,ACH-000998,4.169404


In [26]:
# 1. Одна строка на модель с измеренным эффектом нокаута KRAS
KRAS_effect = COAD_CRISPRGeneEffect[
    ["ModelID", "KRAS (3845)"]
].rename(columns={"KRAS (3845)": "KRAS_GeneEffect"})

# 2. Добавляем экспрессию
KRAS_data = KRAS_effect.merge(
    KRAS_expression,
    on="ModelID",
    how="inner",
    validate="one_to_one"
)

# 3. Добавляем конкретные варианты белка и hotspot-признак
KRAS_data = KRAS_data.merge(
    KRAS_changes, on="ModelID", how="left", validate="one_to_one"
)
hotspot_ids = KRAS_hotspot["ModelID"].unique()

KRAS_data["KRAS_hotspot"] = (
    KRAS_data["ModelID"].isin(hotspot_ids).astype("Int64")
)

# 4. Проверяем наличие мутационных данных основного профиля
default_mutations = COAD_Mutations.loc[
    COAD_Mutations["IsDefaultEntryForModel"].eq("Yes")
]

has_mutation_data = KRAS_data["ModelID"].isin(
    default_mutations["ModelID"]
)

# Нет мутационных записей — статус неизвестен, а не 0
KRAS_data.loc[~has_mutation_data, ["KRAS_hotspot", "KRAS_protein_changes"]] = pd.NA

# Если аннотация hotspot неизвестна и нет подтверждённого hotspot, не ставим 0.
unknown_ids = kras_variants.loc[kras_variants["is_hotspot"].isna(), "ModelID"]
unknown_status = (
    KRAS_data["ModelID"].isin(unknown_ids)
    & ~KRAS_data["ModelID"].isin(hotspot_ids)
)
KRAS_data.loc[unknown_status, "KRAS_hotspot"] = pd.NA


display(KRAS_data)
KRAS_data.to_csv(pilot_dir / "KRAS_data.csv", index=False)

,ModelID,KRAS_GeneEffect,KRAS_expression,KRAS_protein_changes,KRAS_hotspot
0,ACH-000552,-0.468224,4.958708,NaN,0
1,ACH-000958,-0.526112,4.672029,NaN,0
2,ACH-001061,-1.514492,5.191002,p.G13D,1
3,ACH-002669,-0.349134,4.760817,NaN,0
4,ACH-000403,-2.140278,4.167262,p.G13D,1
5,ACH-000959,-1.018418,4.796784,NaN,0
6,ACH-000986,-1.145164,4.704789,NaN,0
7,ACH-000943,-0.521471,4.409235,NaN,0
8,ACH-000963,-0.746334,4.679277,NaN,0
9,ACH-001399,-3.237185,5.129032,p.G12V,1


## Подробные варианты KRAS: очистка и читаемые обозначения

Сохраняем **все варианты KRAS основного профиля**, а не только hotspot. Одна строка — вариант в конкретном секвенировании. Два разных варианта одной модели — не дубль.

Исходные значения сохраняются в `KRAS_variant_details_raw`. В `KRAS_variant_details` приводим строки, числа и флаги к явным типам. Удаляем только полностью совпадающие строки. Конфликтующие записи одного варианта не объединяем молча.

Не заполняем неизвестные AF и аннотации нулями; не удаляем строки по произвольному порогу глубины. Различия AF и AltCount/DP сохраняем: это разные исходные оценки. Сведения о вариантах не доказывают функциональный эффект или долю мутантных клеток.

### Mutations

In [27]:
columns = [
    "ModelID",
    "SequencingID", "ModelConditionID",
    "Chrom", "Pos", "Ref", "Alt",
    "DNAChange", "ProteinChange", "EnsemblFeatureID",
    "VariantType", "VariantInfo",
    "AF", "DP", "RefCount", "AltCount",
    "Hotspot", "LikelyLoF",
]

cond = (
    COAD_Mutations["IsDefaultEntryForModel"].eq("Yes")
    & COAD_Mutations["HugoSymbol"].eq("KRAS")
)

KRAS_variant_details = COAD_Mutations.loc[cond, columns].copy()

display(KRAS_variant_details.head())

,ModelID,SequencingID,ModelConditionID,Chrom,Pos,Ref,Alt,DNAChange,ProteinChange,EnsemblFeatureID,VariantType,VariantInfo,AF,DP,RefCount,AltCount,Hotspot,LikelyLoF
11440,ACH-002660,CDS-TzzXbi,MC-002660-GpFz,chr12,25225713,T,A,ENST00000311936.8:c.351A>T,p.K117N,ENST00000311936,SNV,missense_variant,0.808,74,13,61,True,False
11441,ACH-000249,CDS-6Ck9Ea,MC-000249-ZFHF,chr12,25227341,T,G,ENST00000311936.8:c.183A>C,p.Q61H,ENST00000311936,SNV,missense_variant,0.379,28,18,10,True,False
11442,ACH-000680,CDS-FRVLOG,MC-000680-f7dS,chr12,25227342,T,A,ENST00000311936.8:c.182A>T,p.Q61L,ENST00000311936,SNV,missense_variant,0.667,29,9,20,True,False
11443,ACH-000249,CDS-6Ck9Ea,MC-000249-ZFHF,chr12,25245345,C,T,ENST00000311936.8:c.40G>A,p.V14I,ENST00000311936,SNV,missense_variant,0.452,34,19,15,False,False
11444,ACH-000950,CDS-2s2SzB,MC-000950-Pk7o,chr12,25245347,C,T,ENST00000311936.8:c.38G>A,p.G13D,ENST00000311936,SNV,missense_variant,0.728,56,15,41,True,False


## Additional data for KRAS

Эти столбцы описывают модель и пациента. Они помогают искать подгруппы и возможные причины различий в CRISPR-ответе. `ModelID`, `PatientID` и `CellLineName` служат для идентификации и группировки; их не следует автоматически подавать в ML как обычные признаки.

Пропуск не означает отрицательный ответ. Например, пустой `Stage` означает, что стадия неизвестна, а не раннюю стадию. В текущем COAD-наборе некоторые клинические столбцы почти или полностью пусты, поэтому сначала оцениваем их заполненность.

In [28]:
model_columns = [
    "ModelID", "PatientID", "CellLineName",
    "PrimaryOrMetastasis", "SampleCollectionSite", 
    "Age", "Sex", "GrowthPattern",
]
model_metadata = COAD_Model[model_columns].copy()
assert model_metadata["ModelID"].notna().all()
assert model_metadata["ModelID"].is_unique
model_metadata["Age"] = pd.to_numeric(model_metadata["Age"], errors="coerce")

KRAS_data_expanded = KRAS_data.merge(
    model_metadata, on="ModelID", how="left", validate="one_to_one"
)

metadata_columns = [c for c in model_columns if c != "ModelID"]
metadata_report = pd.DataFrame({
    "dtype": KRAS_data_expanded[metadata_columns].dtypes.astype(str),
    "filled_n": KRAS_data_expanded[metadata_columns].notna().sum(),
    "missing_n": KRAS_data_expanded[metadata_columns].isna().sum(),
    "missing_percent": KRAS_data_expanded[metadata_columns].isna().mean().mul(100).round(1),
    "unique_n": KRAS_data_expanded[metadata_columns].nunique(dropna=True),
}).sort_values(["missing_percent", "unique_n"])

display(metadata_report)

# Категории, которые реально встречаются в 46 моделях анализа.
categorical_metadata = [
 "PrimaryOrMetastasis", "SampleCollectionSite",
    "Sex", "GrowthPattern",
]
for column in categorical_metadata:
    print(f"\n{column}:")
    display(KRAS_data_expanded[column].value_counts(dropna=False).rename("n_models").to_frame())


KRAS_data_expanded.to_csv(pilot_dir / "KRAS_data_expanded.csv", index=False)


,dtype,filled_n,missing_n,missing_percent,unique_n
GrowthPattern,str,46,0,0.0,2
Sex,str,46,0,0.0,3
SampleCollectionSite,str,46,0,0.0,9
PatientID,str,46,0,0.0,45
CellLineName,str,46,0,0.0,46
PrimaryOrMetastasis,str,45,1,2.2,2
Age,float64,38,8,17.4,29



PrimaryOrMetastasis:


,n_models
PrimaryOrMetastasis,
Primary,33
Metastatic,12
NaN,1



SampleCollectionSite:


,n_models
SampleCollectionSite,
large_intestine,19
Colon,17
lymph_node,3
ascites,2
ovary,1
lung,1
pericardial_effusion,1
abdomen,1
liver,1



Sex:


,n_models
Sex,
Male,25
Female,15
Unknown,6



GrowthPattern:


,n_models
GrowthPattern,
Adherent,39
Mixed,7
